# GNN Robustness — FGSM / PGD / Node Injection (Colab)

Runs **FGSM**, **PGD**, and **Node-Injection** evasion attacks against trained GNN
checkpoints on **Elliptic** and **Elliptic++ (actors)**, then aggregates the
results into a paper-ready CSV plus plots / LaTeX tables.

Models supported: `gcn`, `gat`, `graphsage`, `chronowave_gnn` (static) and
`recgnn`, `evolvegcn_o`, `cosemignn` (temporal). The notebook drives
`scripts/run_sweep.py`, which dispatches to the model-specific attack drivers.

**Setup assumptions**
- The repo is cloned from GitHub into `/content/<REPO_DIRNAME>`.
- Raw data lives in your **Google Drive** at `My Drive/data/` and mirrors the
  repo `data/` layout (i.e. `data/raw/elliptic/*.csv` and
  `data/raw/ellipticpp/actors/*.csv`). It is symlinked into the repo's `data/`.


## Install PyTorch Geometric and PyTorch Geometric Temporal

In [2]:
# ============================================================
# Colab setup: PyTorch Geometric + PyTorch Geometric Temporal
# ============================================================

import sys
import subprocess
import torch

def pip_install(args):
    print("$ " + " ".join(map(str, [sys.executable, "-m", "pip", "install", *args])))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

print("torch =", torch.__version__, "| cuda =", torch.version.cuda)

# For your current Colab:
# torch = 2.10.0+cu128
# cuda = 12.8
PYG_WHEEL_URL = "https://data.pyg.org/whl/torch-2.10.0+cu128.html"

# ------------------------------------------------------------
# 1. Install PyG compiled dependencies
# ------------------------------------------------------------
try:
    import torch_scatter
    import torch_sparse
    import pyg_lib
    print("PyG compiled dependencies already installed.")
except Exception:
    pip_install([
        "pyg_lib",
        "torch_scatter",
        "torch_sparse",
        "-f",
        PYG_WHEEL_URL
    ])

# ------------------------------------------------------------
# 2. Install torch-geometric
# ------------------------------------------------------------
try:
    import torch_geometric
    print("torch_geometric already installed:", torch_geometric.__version__)
except Exception:
    pip_install(["torch-geometric"])

# ------------------------------------------------------------
# 3. Install torch-geometric-temporal
# ------------------------------------------------------------
try:
    import torch_geometric_temporal
    print("torch_geometric_temporal already installed.")
except Exception:
    pip_install(["torch-geometric-temporal", "--no-deps"])

# ------------------------------------------------------------
# 4. Install pyyaml, used by run_sweep.py
# ------------------------------------------------------------
try:
    import yaml
    print("pyyaml already installed.")
except Exception:
    pip_install(["-U", "pyyaml"])

print("\nSetup finished.")
print("If this is the first time you installed these packages in this session, click:")
print("Runtime → Restart session")

torch = 2.10.0+cu128 | cuda = 12.8
$ /usr/bin/python3 -m pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
$ /usr/bin/python3 -m pip install torch-geometric
$ /usr/bin/python3 -m pip install torch-geometric-temporal --no-deps
pyyaml already installed.

Setup finished.
If this is the first time you installed these packages in this session, click:
Runtime → Restart session


# Restart the session: Runtime -> Restart Session

In [2]:
# ------------------------------------------------------------
# Test cell
# ------------------------------------------------------------

import torch
import torch_geometric
import torch_geometric_temporal
import yaml

print("torch:", torch.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("torch_geometric_temporal works")
print("pyyaml works")

torch: 2.10.0+cu128
torch_geometric: 2.7.0
torch_geometric_temporal works
pyyaml works


## 0) Experiment configuration

In [3]:
#@title Experiment configuration

# ---- Repo (GitHub clone) ----
REPO_URL     = "https://github.com/koshimbetovv/gnn-robustness-blockchain-research.git"  #@param {type:"string"}
REPO_BRANCH  = "main"                                                                     #@param {type:"string"}
REPO_DIRNAME = "gnn-robustness-blockchain-research"                                        #@param {type:"string"}

# ---- Google Drive data location ----
# Mounts My Drive at /content/drive and expects the data tree at:
#   /content/drive/MyDrive/data/raw/elliptic/...
#   /content/drive/MyDrive/data/raw/ellipticpp/actors/...
DRIVE_DATA_REL = "data"  #@param {type:"string"}

# ---- Datasets to sweep over ----
DATASETS = ["elliptic", "ellipticpp_actors"]  #@param

# ---- Models to attack (must already have checkpoints under models/Elliptic[++]) ----
MODELS = [
    "gcn", "gat", "graphsage", "chronowave_gnn",
    "recgnn", "evolvegcn_o", "cosemignn",
]

# ---- Attack-selection randomness ----
ATTACK_SEEDS = [0, 1, 2]


# ---- Target-selection knobs (forwarded to attack scripts that have them) ----
SPLIT = "test"
ATTACK_ONLY_ILLICIT = True
ONLY_CLEAN_CORRECT = True
ATTACK_FRACTION = 1.0

# ---- Output ----
ARTIFACTS_DIR = "artifacts"

# ---- Raw data file expectations (relative to the repo root) ----
ELLIPTIC_RAW_DIR = "data/raw/elliptic"
ELLIPTIC_REQUIRED_FILES = [
    "elliptic_txs_features.csv",
    "elliptic_txs_classes.csv",
    "elliptic_txs_edgelist.csv",
]
ELLIPTICPP_RAW_DIR = "data/raw/ellipticpp/actors"
ELLIPTICPP_REQUIRED_FILES = [
    "wallets_features.csv",
    "wallets_classes.csv",
    "AddrAddr_edgelist.csv",
]


## 1) Mount Google Drive and clone the repo

In [6]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

def run(cmd, cwd=None):
    print("$ " + " ".join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)), cwd=cwd)

# --- Mount Drive ---
from google.colab import drive  # type: ignore
drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ABS = Path("/content/drive/MyDrive") / DRIVE_DATA_REL
assert DRIVE_DATA_ABS.exists(), (
    f"Expected Drive data folder at {DRIVE_DATA_ABS}. "
    f"Place your data tree (raw/elliptic, raw/ellipticpp/actors, ...) there."
)
print("Drive data:", DRIVE_DATA_ABS)

# --- Clone the repo into /content ---
WORK = Path("/content")
REPO_ROOT = (WORK / REPO_DIRNAME).resolve()
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

print("REPO_ROOT =", REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive data: /content/drive/MyDrive/data
$ git clone --depth 1 --branch main https://github.com/koshimbetovv/gnn-robustness-blockchain-research.git /content/gnn-robustness-blockchain-research
REPO_ROOT = /content/gnn-robustness-blockchain-research


## 2) Wire the repo's `data/` directory to Google Drive

The repo's loaders read raw CSVs from `<REPO_ROOT>/data/...`. We replace that
folder with a symlink to your Drive `data/` folder so loads happen straight
from Drive (no copies, no re-uploads).


In [7]:
repo_data = REPO_ROOT / "data"
if repo_data.is_symlink() or repo_data.exists():
    if repo_data.is_symlink():
        repo_data.unlink()
    else:
        shutil.rmtree(repo_data)

repo_data.symlink_to(DRIVE_DATA_ABS, target_is_directory=True)
print(f"Linked {repo_data} -> {DRIVE_DATA_ABS}")

# Sanity check: required files visible through the symlink
def check(rel_dir, required, label):
    rd = REPO_ROOT / rel_dir
    missing = [f for f in required if not (rd / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"{label}: missing files {missing} under {rd}. "
            f"Put them in your Drive at {DRIVE_DATA_ABS}/{rel_dir.split('data/',1)[-1]}/"
        )
    print(f"✓ {label}: all raw files present at {rd}")

if "elliptic" in DATASETS:
    check(ELLIPTIC_RAW_DIR, ELLIPTIC_REQUIRED_FILES, "Elliptic")
if "ellipticpp_actors" in DATASETS:
    check(ELLIPTICPP_RAW_DIR, ELLIPTICPP_REQUIRED_FILES, "Elliptic++ actors")


Linked /content/gnn-robustness-blockchain-research/data -> /content/drive/MyDrive/data
✓ Elliptic: all raw files present at /content/gnn-robustness-blockchain-research/data/raw/elliptic
✓ Elliptic++ actors: all raw files present at /content/gnn-robustness-blockchain-research/data/raw/ellipticpp/actors


## 3) Wire the repo's `models/` directory to Google Drive

Attacks need pretrained checkpoints from `models/Elliptic/` and/or `models/Elliptic++`.
We replace the repo's `models/` folder with a symlink to your Drive `models/` folder so checkpoints are loaded directly from Drive.

In [8]:
DRIVE_MODELS_ABS = Path("/content/drive/MyDrive") / "models"
assert DRIVE_MODELS_ABS.exists(), (
    f"Expected Drive models folder at {DRIVE_MODELS_ABS}. "
    f"Please create a 'models' folder in your Google Drive and place your trained model checkpoints there."
)
print("Drive models:", DRIVE_MODELS_ABS)

repo_models = REPO_ROOT / "models"
if repo_models.is_symlink() or repo_models.exists():
    if repo_models.is_symlink():
        repo_models.unlink()
    else:
        shutil.rmtree(repo_models)

repo_models.symlink_to(DRIVE_MODELS_ABS, target_is_directory=True)
print(f"Linked {repo_models} -> {DRIVE_MODELS_ABS}")

Drive models: /content/drive/MyDrive/models
Linked /content/gnn-robustness-blockchain-research/models -> /content/drive/MyDrive/models


## 4) Verify trained checkpoints

Attacks need pretrained checkpoints in `models/Elliptic/` and/or `models/Elliptic++/`.
This notebook does not train — use the per-model `notebooks/colab_training/*.ipynb`
notebooks to produce checkpoints if needed. Checkpoints are committed to the
repo (LFS or otherwise), so cloning above should already provide them.


In [9]:
DATASET_TO_MODELDIR = {
    "elliptic": "models/Elliptic",
    "ellipticpp_actors": "models/Elliptic++",
}

def latest_ckpt(model_name: str, model_dir_rel: str):
    md_abs = REPO_ROOT / model_dir_rel
    if not md_abs.exists():
        return None
    runs = sorted([p for p in md_abs.glob(f"{model_name}_*") if (p / "model.pt").exists()])
    return runs[-1] if runs else None

ok = True
for ds in DATASETS:
    md_rel = DATASET_TO_MODELDIR[ds]
    print(f"\n[{ds} -> {md_rel}]")
    for m in MODELS:
        ck = latest_ckpt(m, md_rel)
        if ck is None:
            print(f"  ✗ {m}: NO checkpoint")
            ok = False
        else:
            print(f"  ✓ {m}: {ck.relative_to(REPO_ROOT)}")
if not ok:
    print("\nWARNING: some checkpoints are missing. The sweep will skip those (model, dataset) pairs at runtime.")



[elliptic -> models/Elliptic]
  ✓ gcn: models/Elliptic/gcn_seed46_20260330_142208
  ✓ gat: models/Elliptic/gat_seed44_20260330_152533
  ✓ graphsage: models/Elliptic/graphsage_seed44_20260330_145528
  ✓ chronowave_gnn: models/Elliptic/chronowave_gnn_seed42_seed42_seed46_20260330_212001
  ✓ recgnn: models/Elliptic/recgnn_seed43_20260330_174916
  ✓ evolvegcn_o: models/Elliptic/evolvegcn_o_seed43_20260331_133701
  ✓ cosemignn: models/Elliptic/cosemignn_exact_20260330_194737

[ellipticpp_actors -> models/Elliptic++]
  ✓ gcn: models/Elliptic++/gcn_ellipticpp_actors_seed42_20260401_012929
  ✓ gat: models/Elliptic++/gat_ellipticpp_actors_seed42_20260401_002809
  ✓ graphsage: models/Elliptic++/graphsage_ellipticpp_actors_seed42_20260401_081237
  ✓ chronowave_gnn: models/Elliptic++/chronowave_gnn_ellipticpp_actors_seed46_20260331_205317
  ✓ recgnn: models/Elliptic++/recgnn_ellipticpp_actors_seed46_20260401_084439
  ✓ evolvegcn_o: models/Elliptic++/evolvegcn_o_ellipticpp_actors_seed46_20260401_1

## 5) Define the attack sweep grid

In [10]:
# Per-attack hyperparameter grids. Edit budgets here.
SWEEP = {
    "fgsm": {
        "EPS": [0.005, 0.01, 0.03, 0.05, 0.08, 0.1, 0.2],
    },
    "pgd": {
        "EPS": [0.005, 0.01, 0.03, 0.05, 0.1],
        "STEPS": [10, 20, 30],
        "ALPHA": ["auto"],   # auto => 2*eps/steps inside run_sweep.py
        "RANDOM_START": [True],
    },
    "node_injection": {
        "N_INJECT": [1, 5, 10, 15, 20],
        "EDGES_PER_INJECTED": [5, 10, 20, 30, 40, 50],
        "EPS": [0.01, 0.03, 0.05, 0.1],
        "STEPS": [30],
        "ALPHA": [0.01],
        "RANDOM_START": [True],
        "INIT": ["mean"],
        "CONNECT_STRATEGY": ["round_robin"],
    },
}
ATTACKS = ["fgsm", "pgd", "node_injection"]


## 6) Run the sweep

In [11]:
import yaml

def write_sweep_yaml(out_path: Path):
    sweeps = []
    for atk in ATTACKS:
        sweeps.append({"attack": atk, "params": SWEEP[atk]})
    cfg = {
        "global": {
            "split": SPLIT,
            "attack_only_illicit": bool(ATTACK_ONLY_ILLICIT),
            "only_clean_correct": bool(ONLY_CLEAN_CORRECT),
            "attack_fraction": float(ATTACK_FRACTION),
            "seeds": [int(s) for s in ATTACK_SEEDS],
            "models": list(MODELS),
            "datasets": list(DATASETS),
        },
        "sweeps": sweeps,
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return out_path

cfg_path = REPO_ROOT / "config" / "experiments_robustness_3atk.yaml"
write_sweep_yaml(cfg_path)
print("Wrote:", cfg_path)

run([sys.executable, str(REPO_ROOT / "scripts" / "run_sweep.py"), str(cfg_path)],
    cwd=str(REPO_ROOT))


Wrote: /content/gnn-robustness-blockchain-research/config/experiments_robustness_3atk.yaml
$ /usr/bin/python3 /content/gnn-robustness-blockchain-research/scripts/run_sweep.py /content/gnn-robustness-blockchain-research/config/experiments_robustness_3atk.yaml



KeyboardInterrupt



## 7) Load the consolidated results

`run_sweep.py` writes `attacks/results_summary.csv` covering every attack run.


In [ ]:
import pandas as pd, numpy as np

SUMMARY_CSV = REPO_ROOT / "attacks" / "results_summary.csv"
assert SUMMARY_CSV.exists(), "Missing attacks/results_summary.csv"
df = pd.read_csv(SUMMARY_CSV)
print("rows:", len(df), "cols:", len(df.columns))
df.head(2)


## 8) Normalize columns

Static and temporal scripts emit slightly different metric paths. We collapse
both into a tidy frame keyed by `(attack, model, dataset, seed, *budgets)`.


In [ ]:
def col(df, name):
    return df[name] if name in df.columns else pd.Series([np.nan] * len(df))

def first(*series_list):
    out = None
    for s in series_list:
        if isinstance(s, float) and np.isnan(s):
            continue
        out = s if out is None else out.where(~out.isna(), s)
    return out

out = pd.DataFrame()
out["run_dir"]   = df["run_dir"]
out["attack"]    = col(df, "config.attack").str.lower()
out["model"]     = col(df, "config.model_name")
out["dataset"]   = col(df, "config.dataset")
out["seed"]      = col(df, "config.target_selection.seed")

# Attack budgets (sparse — only some attacks set each)
out["eps"]               = col(df, "config.attack_params.eps")
out["pgd_steps"]         = col(df, "config.attack_params.steps")
out["pgd_alpha"]         = col(df, "config.attack_params.alpha")
out["random_start"]      = col(df, "config.attack_params.random_start")
out["n_inject"]          = col(df, "config.attack_params.n_inject")
out["edges_per_injected"] = col(df, "config.attack_params.edges_per_injected")
out["init"]              = col(df, "config.attack_params.init")
out["connect_strategy"]  = col(df, "config.attack_params.connect_strategy")

# Static metrics layout
f1_pos_clean_s   = col(df, "metrics.classification.aggregate.f1_pos.clean")
f1_pos_adv_s     = col(df, "metrics.classification.aggregate.f1_pos.adv")
f1_pos_drop_s    = col(df, "metrics.classification.aggregate.f1_pos.drop")
recall_pos_clean_s = col(df, "metrics.classification.aggregate.recall_pos.clean")
recall_pos_adv_s   = col(df, "metrics.classification.aggregate.recall_pos.adv")
f1_macro_clean_s = col(df, "metrics.classification.aggregate.f1_macro.clean")
f1_macro_adv_s   = col(df, "metrics.classification.aggregate.f1_macro.adv")
roc_clean_s      = col(df, "metrics.classification.aggregate.roc_auc.clean")
roc_adv_s        = col(df, "metrics.classification.aggregate.roc_auc.adv")
asr_s            = col(df, "metrics.attack_effect.asr.value")
asr_pos_s        = col(df, "metrics.attack_effect.asr_pos_neg.asr_pos")
asr_neg_s        = col(df, "metrics.attack_effect.asr_pos_neg.asr_neg")
conf_drop_s      = col(df, "metrics.attack_effect.mean_confidence_drop.value")
attack_time_s    = col(df, "metrics.attack_effect.attack_time_seconds")

# Temporal metrics layout (concat across timesteps)
f1_pos_clean_t   = col(df, "metrics.classification.aggregate_concat.f1_pos_clean")
f1_pos_adv_t     = col(df, "metrics.classification.aggregate_concat.f1_pos_adv")
f1_pos_drop_t    = col(df, "metrics.classification.aggregate_concat.f1_pos_drop")
recall_pos_clean_t = col(df, "metrics.classification.aggregate_concat.recall_pos_clean")
recall_pos_adv_t   = col(df, "metrics.classification.aggregate_concat.recall_pos_adv")
f1_macro_clean_t = col(df, "metrics.classification.aggregate_concat.f1_macro_clean")
f1_macro_adv_t   = col(df, "metrics.classification.aggregate_concat.f1_macro_adv")
roc_clean_t      = col(df, "metrics.classification.aggregate_concat.roc_auc_clean")
roc_adv_t        = col(df, "metrics.classification.aggregate_concat.roc_auc_adv")
asr_t            = col(df, "metrics.attack_effect.aggregate_concat.asr")
asr_pos_t        = col(df, "metrics.attack_effect.aggregate_concat.asr_pos")
asr_neg_t        = col(df, "metrics.attack_effect.aggregate_concat.asr_neg")
conf_drop_t      = col(df, "metrics.attack_effect.aggregate_concat.mean_confidence_drop")
attack_time_t    = col(df, "metrics.attack_effect.attack_time_seconds")

out["f1_pos_clean"]    = first(f1_pos_clean_s, f1_pos_clean_t)
out["f1_pos_adv"]      = first(f1_pos_adv_s, f1_pos_adv_t)
out["f1_pos_drop"]     = first(f1_pos_drop_s, f1_pos_drop_t)
out["recall_pos_clean"] = first(recall_pos_clean_s, recall_pos_clean_t)
out["recall_pos_adv"]   = first(recall_pos_adv_s, recall_pos_adv_t)
out["f1_macro_clean"]  = first(f1_macro_clean_s, f1_macro_clean_t)
out["f1_macro_adv"]    = first(f1_macro_adv_s, f1_macro_adv_t)
out["roc_auc_clean"]   = first(roc_clean_s, roc_clean_t)
out["roc_auc_adv"]     = first(roc_adv_s, roc_adv_t)
out["asr"]             = first(asr_s, asr_t)
out["asr_pos"]         = first(asr_pos_s, asr_pos_t)
out["asr_neg"]         = first(asr_neg_s, asr_neg_t)
out["conf_drop"]       = first(conf_drop_s, conf_drop_t)
out["attack_time_s"]   = first(attack_time_s, attack_time_t)

# normalize attack name (some scripts emit "NodeInjectionEvasion")
out["attack"] = out["attack"].str.lower().replace({
    "fgsm": "fgsm", "pgd": "pgd",
    "nodeinjectionevasion": "node_injection",
    "node_injection": "node_injection",
})
print(out["attack"].value_counts(dropna=False))
out.head(3)


## 9) Plots — paper-ready PNG + PDF per (model, dataset)

In [ ]:
import matplotlib.pyplot as plt

def savefig(path_base: Path):
    path_base.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(path_base.with_suffix('.png')), dpi=300, bbox_inches='tight')
    plt.savefig(str(path_base.with_suffix('.pdf')), bbox_inches='tight')
    plt.close()

def line_plot(sub, x, y, group_cols, title, xlabel, ylabel, out_base):
    sub = sub.dropna(subset=[x, y])
    if sub.empty:
        return
    keys = [x] + list(group_cols)
    agg = sub.groupby(keys, dropna=False)[y].agg(['mean', 'std', 'count']).reset_index()
    plt.figure(figsize=(5.5, 3.5))
    if group_cols:
        for gvals, gdf in agg.groupby(list(group_cols), dropna=False):
            if not isinstance(gvals, tuple):
                gvals = (gvals,)
            label = ", ".join(f"{c}={v}" for c, v in zip(group_cols, gvals))
            gdf = gdf.sort_values(x)
            plt.plot(gdf[x], gdf['mean'], marker='o', label=label)
            if (gdf['count'] > 1).any():
                plt.fill_between(gdf[x],
                                 gdf['mean'] - gdf['std'].fillna(0),
                                 gdf['mean'] + gdf['std'].fillna(0),
                                 alpha=0.2)
        plt.legend(fontsize=8)
    else:
        agg = agg.sort_values(x)
        plt.plot(agg[x], agg['mean'], marker='o')
        if (agg['count'] > 1).any():
            plt.fill_between(agg[x],
                             agg['mean'] - agg['std'].fillna(0),
                             agg['mean'] + agg['std'].fillna(0),
                             alpha=0.2)
    plt.title(title, fontsize=10)
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    savefig(out_base)

ART = REPO_ROOT / ARTIFACTS_DIR
ART.mkdir(parents=True, exist_ok=True)

for (ds, model), sub in out.groupby(["dataset", "model"]):
    base = ART / ds / model / "plots"

    fgsm = sub[sub["attack"] == "fgsm"]
    line_plot(fgsm, "eps", "f1_pos_adv", [], f"FGSM | {model} on {ds} — F1_pos vs eps",
              "eps", "F1_pos (adv)", base / "fgsm_f1pos_vs_eps")
    line_plot(fgsm, "eps", "asr", [], f"FGSM | {model} on {ds} — ASR vs eps",
              "eps", "ASR", base / "fgsm_asr_vs_eps")

    pgd = sub[sub["attack"] == "pgd"]
    line_plot(pgd, "eps", "f1_pos_adv", ["pgd_steps"],
              f"PGD | {model} on {ds} — F1_pos vs eps", "eps", "F1_pos (adv)",
              base / "pgd_f1pos_vs_eps")
    line_plot(pgd, "eps", "asr", ["pgd_steps"],
              f"PGD | {model} on {ds} — ASR vs eps", "eps", "ASR",
              base / "pgd_asr_vs_eps")

    inj = sub[sub["attack"] == "node_injection"]
    inj05 = inj[np.isclose(inj["eps"].astype(float), 0.05)]
    line_plot(inj05, "n_inject", "asr", ["edges_per_injected"],
              f"NodeInj | {model} on {ds} — ASR vs n_inject (eps=0.05)",
              "n_inject", "ASR",
              base / "inj_asr_vs_ninject")
    line_plot(inj05, "n_inject", "f1_pos_adv", ["edges_per_injected"],
              f"NodeInj | {model} on {ds} — F1_pos vs n_inject (eps=0.05)",
              "n_inject", "F1_pos (adv)",
              base / "inj_f1pos_vs_ninject")

print("Plots written under:", ART)


## 10) Tables — CSV + LaTeX

In [ ]:
def to_latex(df_, path: Path, caption: str, label: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    tex = df_.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, (float, np.floating)) else str(x),
        escape=False,
    )
    insert = "\\toprule\n" + f"\\caption{{{caption}}}\\label{{{label}}}\\\\\n"
    tex = tex.replace("\\toprule", insert, 1)
    path.write_text(tex, encoding="utf-8")

agg_cols = {
    "f1_pos_clean": "mean", "f1_pos_adv": "mean", "f1_pos_drop": "mean",
    "recall_pos_clean": "mean", "recall_pos_adv": "mean",
    "f1_macro_clean": "mean", "f1_macro_adv": "mean",
    "roc_auc_clean": "mean", "roc_auc_adv": "mean",
    "asr": "mean", "asr_pos": "mean", "asr_neg": "mean",
    "conf_drop": "mean", "attack_time_s": "mean",
}

# 1) Headline robustness summary per (dataset, model, attack, key budgets)
for ds, sub_ds in out.groupby("dataset"):
    for model, sub in sub_ds.groupby("model"):
        tdir = ART / ds / model / "tables"
        tdir.mkdir(parents=True, exist_ok=True)

        for atk, atk_grp in sub.groupby("attack"):
            if atk == "fgsm":
                keys = ["eps"]
            elif atk == "pgd":
                keys = ["eps", "pgd_steps"]
            elif atk == "node_injection":
                keys = ["eps", "n_inject", "edges_per_injected"]
            else:
                continue
            keys = [k for k in keys if k in atk_grp.columns]

            grp = (atk_grp.groupby(keys, dropna=False)
                          .agg(agg_cols).reset_index()
                          .sort_values(keys))
            csv_path = tdir / f"{atk}_summary.csv"
            grp.to_csv(csv_path, index=False)
            to_latex(
                grp, tdir / f"{atk}_summary.tex",
                caption=f"{atk.upper()} robustness — {model} on {ds} (mean over seeds)",
                label=f"tab:{ds}:{model}:{atk}",
            )

        base = sub.groupby("attack")[["f1_pos_clean", "recall_pos_clean", "f1_macro_clean", "roc_auc_clean"]].mean().reset_index()
        base.to_csv(tdir / "clean_baseline.csv", index=False)

# 2) Cross-model comparison per (dataset, attack)
for (ds, atk), sub in out.groupby(["dataset", "attack"]):
    if atk not in ("fgsm", "pgd", "node_injection"):
        continue
    keys = ["model"]
    if atk == "fgsm":
        keys += ["eps"]
    elif atk == "pgd":
        keys += ["eps", "pgd_steps"]
    else:
        keys += ["eps", "n_inject", "edges_per_injected"]
    keys = [k for k in keys if k in sub.columns]

    grp = (sub.groupby(keys, dropna=False)
              .agg(agg_cols).reset_index()
              .sort_values(keys))
    out_dir = ART / ds / "_cross_model" / "tables"
    out_dir.mkdir(parents=True, exist_ok=True)
    grp.to_csv(out_dir / f"{atk}_cross_model.csv", index=False)
    to_latex(grp, out_dir / f"{atk}_cross_model.tex",
             caption=f"{atk.upper()} cross-model — {ds}",
             label=f"tab:{ds}:cross_model:{atk}")

print("Tables written under:", ART)


## 11) Export artifacts (zip + download)

In [13]:
import zipfile

zip_path = REPO_ROOT / f"{ARTIFACTS_DIR}.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in (REPO_ROOT / ARTIFACTS_DIR).rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    if SUMMARY_CSV.exists():
        z.write(SUMMARY_CSV, arcname="attacks/results_summary.csv")

print("Wrote:", zip_path)

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    print("Not on Colab — find the zip at:", zip_path)


NameError: name 'SUMMARY_CSV' is not defined

### Download `/attacks` folder

This cell will create a zip file containing only the `attacks` directory and provide a download link.

In [ ]:
import zipfile
import os
from pathlib import Path
from google.colab import files # For downloading

# Assuming REPO_ROOT is defined from earlier cells
if 'REPO_ROOT' not in locals():
    # Fallback if REPO_ROOT is not yet defined (e.g., if this cell is run out of order)
    REPO_ROOT = (Path('/content') / REPO_DIRNAME).resolve()

attacks_dir = REPO_ROOT / "attacks"
zip_file_name = f"{REPO_DIRNAME}_attacks.zip"
zip_path = REPO_ROOT / zip_file_name

if zip_path.exists():
    zip_path.unlink() # Remove previous zip if it exists

if attacks_dir.exists() and attacks_dir.is_dir():
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for file_path in attacks_dir.rglob("*"):
            if file_path.is_file():
                # Write to zip, keeping the 'attacks/' prefix relative to REPO_ROOT
                z.write(file_path, arcname=str(file_path.relative_to(REPO_ROOT)))
    print(f"Wrote: {zip_path}")

    try:
        files.download(str(zip_path))
    except Exception:
        print("Not on Colab — find the zip at:", zip_path)
else:
    print(f"The '/attacks' directory was not found at {attacks_dir}.")
